# Simplified Face Recognition Attendance System (Jupyter)

This notebook implements a face recognition attendance workflow using:

- OpenCV DNN face detector (Caffe SSD model)
- ArcFace ONNX model (512D embeddings via onnxruntime)
- Cosine-distance matching (vectorized) with an optional approximate prefilter
- Attendance logging to `attendance.csv`

**Design goals (same as the Streamlit version):**
- Reduced complexity and clean flow
- Compare embeddings computed from a **tight face crop** (less background)
- Faster matching using vectorized cosine distance and optional prefilter

## Imports

If you get missing package errors, install them with:
- `uv add opencv-python onnxruntime pandas requests pillow ipywidgets streamlit`

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import requests
from datetime import datetime
from PIL import Image
import onnxruntime as ort

# Optional (for upload widget in Jupyter)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

In [2]:
# =============================================================================
# 1) CONFIG
# =============================================================================

KNOWN_FACES_DIR = "known_Faces"
ATTENDANCE_FILE = "attendance.csv"

ARCFACE_MODEL_PATH = "arcface.onnx"
FACE_PROTO = "deploy.prototxt"
FACE_MODEL = "res10_300x300_ssd_iter_140000.caffemodel"

# Detection / recognition settings
DEFAULT_DETECTION_CONF = 0.80
DEFAULT_COSINE_THRESHOLD = 0.40  # lower = stricter (distance)
MIN_FACE_SIZE_PX = 80            # ignore tiny detections (speed + fewer false positives)

# Matching speed tweaks
USE_PREFILTER = True             # approximate prefilter using a few dims (keeps accuracy mostly OK)
PREFILTER_DIMS = 64              # 32..128 typical; higher = more accurate but slower
PREFILTER_TOPK = 30              # compare full 512D only to top K candidates

In [3]:
# =============================================================================
# 2) FILES
# =============================================================================

os.makedirs(KNOWN_FACES_DIR, exist_ok=True)

if not os.path.exists(ATTENDANCE_FILE):
    pd.DataFrame(columns=["Index", "Name", "Date", "Time"]).to_csv(ATTENDANCE_FILE, index=False)

print("Setup complete.")
print("Known faces dir:", KNOWN_FACES_DIR)
print("Attendance file:", ATTENDANCE_FILE)

Setup complete.
Known faces dir: known_Faces
Attendance file: attendance.csv


## Download models (only if missing)

We download:
- ArcFace ONNX model
- OpenCV DNN face detector prototxt + caffemodel

In [4]:
# =============================================================================
# 3) DOWNLOAD MODELS
# =============================================================================

def _download(url: str, path: str, stream: bool = False):
    r = requests.get(url, allow_redirects=True, stream=stream, timeout=60)
    r.raise_for_status()
    if stream:
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    else:
        with open(path, "wb") as f:
            f.write(r.content)

def download_models() -> bool:
    try:
        if not os.path.exists(ARCFACE_MODEL_PATH):
            print("Downloading ArcFace model...")
            _download(
                "https://huggingface.co/garavv/arcface-onnx/resolve/main/arc.onnx",
                ARCFACE_MODEL_PATH,
                stream=True,
            )

        if not os.path.exists(FACE_PROTO):
            print("Downloading face detector prototxt...")
            _download(
                "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
                FACE_PROTO,
                stream=False,
            )

        if not os.path.exists(FACE_MODEL):
            print("Downloading face detector caffemodel...")
            _download(
                "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
                FACE_MODEL,
                stream=True,
            )

        return True
    except Exception as e:
        print(f"Model download failed: {e}")
        return False

ok = download_models()
assert ok, "Model download failed."
print("Model files ready.")

Model files ready.


## Load models into memory

- OpenCV face detector is loaded once
- ArcFace ONNX session is created with CPU optimization settings

In [5]:
# =============================================================================
# 4) LOAD MODELS
# =============================================================================

def load_models():
    face_net = cv2.dnn.readNetFromCaffe(FACE_PROTO, FACE_MODEL)

    # Faster CPU settings (best effort)
    so = ort.SessionOptions()
    so.intra_op_num_threads = max(1, os.cpu_count() or 1)
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    arc = ort.InferenceSession(
        ARCFACE_MODEL_PATH,
        sess_options=so,
        providers=["CPUExecutionProvider"],
    )
    input_name = arc.get_inputs()[0].name
    return face_net, arc, input_name

face_net, arcface_session, arcface_input_name = load_models()
print("Models loaded.")
print("ArcFace input name:", arcface_input_name)

Models loaded.
ArcFace input name: input_1


## Core functions

Key steps:
1. Detect faces
2. Pick the best face (largest area, then confidence)
3. Tight crop with small padding to reduce background
4. Preprocess and run ArcFace to get a 512D L2-normalized embedding
5. Match with vectorized cosine distance, optional prefilter

In [6]:
# =============================================================================
# 5) CORE FUNCTIONS
# =============================================================================

def detect_faces(image_bgr: np.ndarray, conf_threshold: float):
    """Return list of (box, confidence, area)."""
    h, w = image_bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(image_bgr, 1.0, (300, 300), (104, 177, 123))
    face_net.setInput(blob)
    det = face_net.forward()

    faces = []
    for i in range(det.shape[2]):
        conf = float(det[0, 0, i, 2])
        if conf < conf_threshold:
            continue

        x1, y1, x2, y2 = (det[0, 0, i, 3:7] * np.array([w, h, w, h])).astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        fw, fh = (x2 - x1), (y2 - y1)
        if fw < MIN_FACE_SIZE_PX or fh < MIN_FACE_SIZE_PX:
            continue

        area = fw * fh
        faces.append(((x1, y1, x2, y2), conf, area))

    return faces

def pick_best_face(faces):
    """Pick largest face; tie-break by confidence."""
    if not faces:
        return None
    return max(faces, key=lambda x: (x[2], x[1]))  # (area, confidence)

def crop_face_tight(image_bgr: np.ndarray, box):
    """
    Tight face crop to minimize background.
    Uses small padding to avoid cutting facial contour too aggressively.
    """
    x1, y1, x2, y2 = box
    h, w = image_bgr.shape[:2]

    fw, fh = x2 - x1, y2 - y1
    pad = int(min(fw, fh) * 0.08)  # small padding => less background

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)
    face = image_bgr[y1:y2, x1:x2]
    return face

def preprocess_for_arcface(face_bgr: np.ndarray):
    face = cv2.resize(face_bgr, (112, 112), interpolation=cv2.INTER_AREA)
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face = face.astype(np.float32)
    face = (face - 127.5) / 128.0
    face = np.expand_dims(face, axis=0)  # (1,112,112,3) expected by this model
    return face

def get_embedding(face_bgr: np.ndarray):
    x = preprocess_for_arcface(face_bgr)
    emb = arcface_session.run(None, {arcface_input_name: x})[0].reshape(-1).astype(np.float32)
    n = np.linalg.norm(emb)
    if n > 0:
        emb = emb / n
    return emb  # L2-normalized 512D

def parse_filename(file: str):
    # expected: index_name.jpg (name can contain underscores only if you keep last "_" split)
    parts = file.rsplit("_", 1)
    if len(parts) != 2:
        return None
    index = parts[0]
    name = os.path.splitext(parts[1])[0]
    return index, name

def cosine_distance_vectorized(known_embs: np.ndarray, test_emb: np.ndarray):
    """
    If embeddings are L2-normalized:
      cosine similarity = dot product
      cosine distance = 1 - dot
    """
    sims = known_embs @ test_emb  # (N,)
    dists = 1.0 - sims
    return dists, sims

def load_known_faces_silent():
    """
    Loads known embeddings silently (no per-file prints).
    Returns:
      known_embs: (N,512) float32
      meta: list of (index, name)
      prefilter: (N, PREFILTER_DIMS) float32 or None
    """
    files = [f for f in os.listdir(KNOWN_FACES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    embs = []
    meta = []

    for f in files:
        parsed = parse_filename(f)
        if parsed is None:
            continue
        img = cv2.imread(os.path.join(KNOWN_FACES_DIR, f))
        if img is None:
            continue

        faces = detect_faces(img, conf_threshold=0.5)  # lenient for enrollment
        best = pick_best_face(faces)
        if best is None:
            continue

        face = crop_face_tight(img, best[0])
        if face.size == 0:
            continue

        emb = get_embedding(face)
        if emb is None or emb.shape[0] != 512:
            continue

        embs.append(emb)
        meta.append(parsed)

    if not embs:
        return np.empty((0, 512), dtype=np.float32), [], None

    known = np.vstack(embs).astype(np.float32)

    if USE_PREFILTER:
        pf = known[:, :PREFILTER_DIMS].copy()
        norms = np.linalg.norm(pf, axis=1, keepdims=True)
        pf = pf / np.clip(norms, 1e-12, None)
        return known, meta, pf

    return known, meta, None

def match_face(test_emb: np.ndarray, known_embs: np.ndarray, meta, cosine_thresh: float, prefilter=None):
    if known_embs.shape[0] == 0:
        return None, None

    candidate_idx = None
    if prefilter is not None and known_embs.shape[0] > PREFILTER_TOPK:
        te = test_emb[:PREFILTER_DIMS]
        te = te / max(np.linalg.norm(te), 1e-12)
        d_pf = 1.0 - (prefilter @ te)
        candidate_idx = np.argsort(d_pf)[:PREFILTER_TOPK]
        cand_embs = known_embs[candidate_idx]
    else:
        cand_embs = known_embs

    dists, sims = cosine_distance_vectorized(cand_embs, test_emb)
    j = int(np.argmin(dists))
    best_dist = float(dists[j])
    best_sim = float(sims[j])

    best_global = int(candidate_idx[j]) if candidate_idx is not None else j

    scores = {"cosine_distance": best_dist, "cosine_similarity": best_sim}
    if best_dist < cosine_thresh:
        return meta[best_global], scores
    return None, scores

def mark_attendance(index: str, name: str):
    now = datetime.now()
    date = now.strftime("%Y-%m-%d")
    time = now.strftime("%H:%M:%S")

    df = pd.read_csv(ATTENDANCE_FILE)
    if not ((df["Index"] == index) & (df["Date"] == date)).any():
        df.loc[len(df)] = [index, name, date, time]
        df.to_csv(ATTENDANCE_FILE, index=False)
        return "Attendance marked"
    return "Attendance already marked today"

## Load known faces

Put images in `known_Faces/` named like:

- `1234_JohnDoe.jpg`
- `99_Jane_Smith.png` (index is before the last `_`, name after it)

The notebook will:
- detect the best face in each image
- compute embeddings from tight face crops
- keep `(index, name)` metadata aligned with embeddings

In [7]:
known_embeddings, known_meta, known_prefilter = load_known_faces_silent()

print("Known faces loaded:", known_embeddings.shape[0])
if known_embeddings.shape[0] == 0:
    print("No known faces found. Add images to `known_Faces` as `index_name.jpg`.")

Known faces loaded: 2


## Visualization helpers

We show:
- the tight-cropped detected face
- detection confidence
- matching scores (cosine distance + similarity)

In [8]:
def bgr_to_pil(image_bgr: np.ndarray) -> Image.Image:
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)

def draw_box(image_bgr: np.ndarray, box, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = box
    out = image_bgr.copy()
    cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
    return out

## Recognition + attendance function

This function takes an input BGR image and runs the full pipeline:
1. detect faces
2. choose best face
3. crop tightly
4. compute embedding
5. match against known embeddings
6. if matched, mark attendance

In [9]:
def recognize_and_mark(
    image_bgr: np.ndarray,
    detection_conf: float = DEFAULT_DETECTION_CONF,
    cosine_thresh: float = DEFAULT_COSINE_THRESHOLD,
    show_debug: bool = True,
):
    if known_embeddings.shape[0] == 0:
        raise RuntimeError("No known faces loaded. Add enrollment images and re-run loading cell.")

    faces = detect_faces(image_bgr, conf_threshold=detection_conf)
    best = pick_best_face(faces)

    if best is None:
        if show_debug:
            print("No face detected.")
        return {"status": "no_face"}

    box, conf, _area = best
    face_bgr = crop_face_tight(image_bgr, box)
    emb = get_embedding(face_bgr)

    match, scores = match_face(
        emb,
        known_embeddings,
        known_meta,
        cosine_thresh=cosine_thresh,
        prefilter=known_prefilter,
    )

    result = {
        "status": "ok",
        "detection_conf": float(conf),
        "box": box,
        "face_crop_bgr": face_bgr,
        "scores": scores,
        "match": match,
        "attendance_msg": None,
    }

    if show_debug:
        print(f"Detection confidence: {conf:.2%}")
        print(f"Cosine distance: {scores['cosine_distance']:.4f} (match if < {cosine_thresh})")
        print(f"Cosine similarity: {scores['cosine_similarity']:.4f}")

    if match is None:
        if show_debug:
            print("Not recognized.")
        result["status"] = "not_recognized"
        return result

    index, name = match
    if show_debug:
        print(f"Recognized: {name} ({index})")

    msg = mark_attendance(index, name)
    result["attendance_msg"] = msg
    if show_debug:
        print(msg)

    return result

## Run on a local image file

Set `test_path` to an image path and run.

In [10]:
test_path = ""  # e.g. "test.jpg"

if test_path and os.path.exists(test_path):
    img_bgr = cv2.imread(test_path)
    res = recognize_and_mark(img_bgr, detection_conf=DEFAULT_DETECTION_CONF, cosine_thresh=DEFAULT_COSINE_THRESHOLD)

    if res.get("face_crop_bgr") is not None:
        display(bgr_to_pil(res["face_crop_bgr"]))
else:
    print("Set test_path to a valid image file to run this cell.")

Set test_path to a valid image file to run this cell.


In [11]:
def upload_and_run_widget():
    if not HAVE_WIDGETS:
        print("ipywidgets not available in this environment.")
        return

    uploader = widgets.FileUpload(accept=".jpg,.jpeg,.png", multiple=False)
    det = widgets.FloatSlider(value=DEFAULT_DETECTION_CONF, min=0.3, max=0.95, step=0.05, description="Detect conf")
    thr = widgets.FloatSlider(value=DEFAULT_COSINE_THRESHOLD, min=0.2, max=0.7, step=0.05, description="Cos thr")
    btn = widgets.Button(description="Run recognition", button_style="primary")
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output()
            if not uploader.value:
                print("Upload an image first.")
                return

            # Get bytes
            item = list(uploader.value.values())[0]
            data = item["content"]
            pil = Image.open(io.BytesIO(data)).convert("RGB")
            image_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

            res = recognize_and_mark(image_bgr, detection_conf=float(det.value), cosine_thresh=float(thr.value), show_debug=True)

            if res.get("box") is not None:
                boxed = draw_box(image_bgr, res["box"])
                print("\nFull image with detection box:")
                display(bgr_to_pil(boxed))

            if res.get("face_crop_bgr") is not None:
                print("\nTight face crop used for embedding:")
                display(bgr_to_pil(res["face_crop_bgr"]))

    import io
    btn.on_click(on_click)

    display(widgets.VBox([uploader, det, thr, btn, out]))

upload_and_run_widget()

ipywidgets not available in this environment.


## Attendance records

This reads `attendance.csv` and allows simple filtering in code.

In [12]:
df = pd.read_csv(ATTENDANCE_FILE)
df

,Index,Name,Date,Time


In [13]:
def filter_attendance(date=None, names=None):
    df = pd.read_csv(ATTENDANCE_FILE)
    out = df.copy()

    if date is not None:
        if isinstance(date, datetime):
            date_str = date.strftime("%Y-%m-%d")
        else:
            date_str = str(date)
        out = out[out["Date"] == date_str]

    if names:
        out = out[out["Name"].isin(names)]

    return out

# Example usage:
# filter_attendance(date=datetime.now(), names=["JohnDoe"])
filter_attendance(date=datetime.now())

,Index,Name,Date,Time


In [14]:
script_content = """import os
import cv2
import numpy as np
import pandas as pd
import requests

import streamlit as st
st.set_page_config(page_title="Face Recognition Attendance", layout="wide")
st.title("Face Recognition Attendance")

from datetime import datetime
from PIL import Image
import onnxruntime as ort

KNOWN_FACES_DIR = "known_Faces"
ATTENDANCE_FILE = "attendance.csv"

ARCFACE_MODEL_PATH = "arcface.onnx"
FACE_PROTO = "deploy.prototxt"
FACE_MODEL = "res10_300x300_ssd_iter_140000.caffemodel"

DEFAULT_DETECTION_CONF = 0.80
DEFAULT_COSINE_THRESHOLD = 0.40
MIN_FACE_SIZE_PX = 80

USE_PREFILTER = True
PREFILTER_DIMS = 64
PREFILTER_TOPK = 30

os.makedirs(KNOWN_FACES_DIR, exist_ok=True)

if not os.path.exists(ATTENDANCE_FILE):
    pd.DataFrame(columns=["Index", "Name", "Date", "Time"]).to_csv(ATTENDANCE_FILE, index=False)

@st.cache_resource
def download_models() -> bool:
    def _download(url: str, path: str, stream: bool = False):
        r = requests.get(url, allow_redirects=True, stream=stream, timeout=60)
        r.raise_for_status()
        if stream:
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        else:
            with open(path, "wb") as f:
                f.write(r.content)

    try:
        if not os.path.exists(ARCFACE_MODEL_PATH):
            st.info("Downloading ArcFace model...")
            _download(
                "https://huggingface.co/garavv/arcface-onnx/resolve/main/arc.onnx",
                ARCFACE_MODEL_PATH,
                stream=True,
            )

        if not os.path.exists(FACE_PROTO):
            _download(
                "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
                FACE_PROTO,
                stream=False,
            )

        if not os.path.exists(FACE_MODEL):
            st.info("Downloading face detector model...")
            _download(
                "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
                FACE_MODEL,
                stream=True,
            )

        return True
    except Exception as e:
        st.error(f"Model download failed: {e}")
        return False

if not download_models():
    st.stop()

@st.cache_resource
def load_models():
    face_net = cv2.dnn.readNetFromCaffe(FACE_PROTO, FACE_MODEL)

    so = ort.SessionOptions()
    so.intra_op_num_threads = max(1, os.cpu_count() or 1)
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    arc = ort.InferenceSession(
        ARCFACE_MODEL_PATH,
        sess_options=so,
        providers=["CPUExecutionProvider"],
    )
    input_name = arc.get_inputs()[0].name
    return face_net, arc, input_name

face_net, arcface_session, arcface_input_name = load_models()

def detect_faces(image_bgr: np.ndarray, conf_threshold: float):
    h, w = image_bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(image_bgr, 1.0, (300, 300), (104, 177, 123))
    face_net.setInput(blob)
    det = face_net.forward()

    faces = []
    for i in range(det.shape[2]):
        conf = float(det[0, 0, i, 2])
        if conf < conf_threshold:
            continue

        x1, y1, x2, y2 = (det[0, 0, i, 3:7] * np.array([w, h, w, h])).astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        fw, fh = (x2 - x1), (y2 - y1)
        if fw < MIN_FACE_SIZE_PX or fh < MIN_FACE_SIZE_PX:
            continue

        area = fw * fh
        faces.append(((x1, y1, x2, y2), conf, area))

    return faces

def pick_best_face(faces):
    if not faces:
        return None
    return max(faces, key=lambda x: (x[2], x[1]))

def crop_face_tight(image_bgr: np.ndarray, box):
    x1, y1, x2, y2 = box
    h, w = image_bgr.shape[:2]

    fw, fh = x2 - x1, y2 - y1
    pad = int(min(fw, fh) * 0.08)

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)
    face = image_bgr[y1:y2, x1:x2]
    return face

def preprocess_for_arcface(face_bgr: np.ndarray):
    face = cv2.resize(face_bgr, (112, 112), interpolation=cv2.INTER_AREA)
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face = face.astype(np.float32)
    face = (face - 127.5) / 128.0
    face = np.expand_dims(face, axis=0)
    return face

def get_embedding(face_bgr: np.ndarray):
    x = preprocess_for_arcface(face_bgr)
    emb = arcface_session.run(None, {arcface_input_name: x})[0].reshape(-1).astype(np.float32)
    n = np.linalg.norm(emb)
    if n > 0:
        emb = emb / n
    return emb

def parse_filename(file: str):
    parts = file.rsplit("_", 1)
    if len(parts) != 2:
        return None
    index = parts[0]
    name = os.path.splitext(parts[1])[0]
    return index, name

def cosine_distance_vectorized(known_embs: np.ndarray, test_emb: np.ndarray):
    sims = known_embs @ test_emb
    dists = 1.0 - sims
    return dists, sims

@st.cache_data(show_spinner=False)
def load_known_faces_silent():
    files = [f for f in os.listdir(KNOWN_FACES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    embs = []
    meta = []

    for f in files:
        parsed = parse_filename(f)
        if parsed is None:
            continue
        img = cv2.imread(os.path.join(KNOWN_FACES_DIR, f))
        if img is None:
            continue

        faces = detect_faces(img, conf_threshold=0.5)
        best = pick_best_face(faces)
        if best is None:
            continue

        face = crop_face_tight(img, best[0])
        if face.size == 0:
            continue

        emb = get_embedding(face)
        if emb is None or emb.shape[0] != 512:
            continue

        embs.append(emb)
        meta.append(parsed)

    if not embs:
        return np.empty((0, 512), dtype=np.float32), [], None

    known = np.vstack(embs).astype(np.float32)

    if USE_PREFILTER:
        pf = known[:, :PREFILTER_DIMS].copy()
        norms = np.linalg.norm(pf, axis=1, keepdims=True)
        pf = pf / np.clip(norms, 1e-12, None)
        return known, meta, pf

    return known, meta, None

def match_face(test_emb: np.ndarray, known_embs: np.ndarray, meta, cosine_thresh: float, prefilter=None):
    if known_embs.shape[0] == 0:
        return None, None

    candidate_idx = None
    if prefilter is not None and known_embs.shape[0] > PREFILTER_TOPK:
        te = test_emb[:PREFILTER_DIMS]
        te = te / max(np.linalg.norm(te), 1e-12)
        d_pf = 1.0 - (prefilter @ te)
        candidate_idx = np.argsort(d_pf)[:PREFILTER_TOPK]
        cand_embs = known_embs[candidate_idx]
    else:
        cand_embs = known_embs

    dists, sims = cosine_distance_vectorized(cand_embs, test_emb)
    j = int(np.argmin(dists))
    best_dist = float(dists[j])
    best_sim = float(sims[j])

    best_global = int(candidate_idx[j]) if candidate_idx is not None else j

    scores = {"cosine_distance": best_dist, "cosine_similarity": best_sim}
    if best_dist < cosine_thresh:
        return meta[best_global], scores
    return None, scores

def mark_attendance(index: str, name: str):
    now = datetime.now()
    date = now.strftime("%Y-%m-%d")
    time = now.strftime("%H:%M:%S")

    df = pd.read_csv(ATTENDANCE_FILE)
    if not ((df["Index"] == index) & (df["Date"] == date)).any():
        df.loc[len(df)] = [index, name, date, time]
        df.to_csv(ATTENDANCE_FILE, index=False)
        return "Attendance marked"
    return "Attendance already marked today"



with st.sidebar:
    st.header("Settings")
    detection_conf = st.slider("Detection confidence", 0.3, 0.95, DEFAULT_DETECTION_CONF, 0.05)
    cosine_thresh = st.slider("Cosine distance threshold", 0.2, 0.7, DEFAULT_COSINE_THRESHOLD, 0.05)

    if st.button("Reload known faces"):
        st.cache_data.clear()
        st.rerun()

known_embeddings, known_meta, known_prefilter = load_known_faces_silent()
if known_embeddings.shape[0] == 0:
    st.warning("No known faces found. Add images to `known_Faces` as `index_name.jpg`.")
    st.stop()

mode = st.radio("Input", ["Upload Image", "Camera"], horizontal=True)

image_bgr = None
if mode == "Upload Image":
    file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])
    if file:
        pil = Image.open(file).convert("RGB")
        image_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
else:
    cam = st.camera_input("Capture an image")
    if cam:
        pil = Image.open(cam).convert("RGB")
        image_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

if image_bgr is not None:
    faces = detect_faces(image_bgr, conf_threshold=detection_conf)
    best = pick_best_face(faces)

    if best is None:
        st.error("No face detected.")
    else:
        box, conf, _area = best
        face_bgr = crop_face_tight(image_bgr, box)
        emb = get_embedding(face_bgr)

        match, scores = match_face(
            emb,
            known_embeddings,
            known_meta,
            cosine_thresh=cosine_thresh,
            prefilter=known_prefilter,
        )

        col1, col2 = st.columns([1, 2])

        with col1:
            st.image(cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB), caption="Detected face", use_column_width=True)
            st.caption(f"Detection confidence: {conf:.2%}")

        with col2:
            st.write(f"Cosine distance: `{scores['cosine_distance']:.4f}`  (match if < `{cosine_thresh}`)")
            st.write(f"Cosine similarity: `{scores['cosine_similarity']:.4f}`")

            if match is None:
                st.error("Not recognized.")
            else:
                index, name = match
                st.success(f"Recognized: {name} ({index})")
                msg = mark_attendance(index, name)
                if "already" in msg.lower():
                    st.warning(msg)
                else:
                    st.success(msg)

st.divider()
st.subheader("Attendance Records")

df = pd.read_csv(ATTENDANCE_FILE)
if len(df) == 0:
    st.info("No attendance records yet.")
else:
    col1, col2 = st.columns(2)
    with col1:
        date_filter = st.date_input("Filter by date", value=datetime.now())
    with col2:
        name_filter = st.multiselect("Filter by name", options=sorted(df["Name"].unique().tolist()))

    out = df.copy()
    if date_filter:
        out = out[out["Date"] == date_filter.strftime("%Y-%m-%d")]
    if name_filter:
        out = out[out["Name"].isin(name_filter)]

    st.dataframe(out, use_container_width=True)
    st.download_button(
        "Download CSV",
        data=out.to_csv(index=False).encode("utf-8"),
        file_name=f"attendance_{datetime.now().strftime('%Y%m%d')}.csv",
        mime="text/csv",
    )
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("✅ app.py generated successfully! You can now run:")
print("   streamlit run app.py")


✅ app.py generated successfully! You can now run:
   streamlit run app.py


In [15]:
# Run the Streamlit app with uv environment
# Headless mode prevents Streamlit from prompting for an email address and opening a browser
!uv run streamlit run app.py --server.headless true

2026-06-09 00:10:42.212 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.1.9:8501
  External URL: http://112.134.168.110:8501

2026-06-09 00:11:45.769 The `use_column_width` parameter has been deprecated and will be removed in a future release. Please utilize the `width` parameter instead.
2026-06-09 00:12:00.374 The `use_column_width` parameter has been deprecated and will be removed in a future release. Please utilize the `width` parameter instead.
2026-06-09 00:13:03.489 The `use_column_width` parameter has been deprecated and will be removed in a future release. Please utilize the `width` parameter instead.
2026-06-09 00:13:03.494 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-09 00:13:10.765 P